# 05 — Counterfactual Checks

**FairLens AI / NyayaLens — Person 3 (ML Layer)**

A **counterfactual check** answers: *"If we flip the sensitive attribute
(e.g., Male -> Female) but keep everything else the same, does the model's
prediction change?"*

If the prediction changes for many samples, the model is using proxies
for the sensitive attribute (even though we removed it from features).

### What we do here
1. Rebuild data and model
2. Create counterfactual test set (flip correlated features)
3. Measure prediction flip rate
4. Identify which samples are most affected

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import fetch_openml
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

sns.set_theme(style="whitegrid")
print("Libraries loaded.")

## Step 1 — Rebuild data and model

In [ ]:
raw_df = fetch_openml("adult", version=2, as_frame=True).frame
cleaned_df = raw_df.copy()
for col in cleaned_df.columns:
    if cleaned_df[col].dtype == object:
        cleaned_df = cleaned_df[cleaned_df[col] != "?"]
cleaned_df = cleaned_df.dropna()

income_labels = cleaned_df["class"].apply(
    lambda v: 1 if ">50K" in str(v) else 0
)
gender_sensitive = cleaned_df["sex"].copy()
feature_df = cleaned_df.drop(columns=["sex", "race", "fnlwgt", "class"])
feature_df = pd.get_dummies(feature_df, drop_first=True)

features_train, features_test, labels_train, labels_test = train_test_split(
    feature_df, income_labels, test_size=0.2, random_state=42
)

scaler = StandardScaler()
features_train = pd.DataFrame(
    scaler.fit_transform(features_train),
    columns=features_train.columns, index=features_train.index
)
features_test = pd.DataFrame(
    scaler.transform(features_test),
    columns=features_test.columns, index=features_test.index
)

gender_test = gender_sensitive.loc[features_test.index]

baseline_model = LogisticRegression(max_iter=5000, random_state=42)
baseline_model.fit(features_train, labels_train)
original_predictions = baseline_model.predict(features_test)

print(f"Model trained. Test set size: {features_test.shape[0]}")

## Step 2 — Create counterfactual test set

Even though `sex` is not in our feature matrix, the model might use
**proxy features** that are correlated with sex. For example:
- `relationship_Wife` / `relationship_Husband` are strongly gender-correlated
- `marital-status_Married-civ-spouse` correlates with gender

In a counterfactual check, we flip these proxy features to simulate
"what if this person were the other gender?" and see if the prediction changes.

Note: This is a simplified version. A full counterfactual analysis would
use more sophisticated methods, but this gives us a directional signal.

In [ ]:
# Find relationship proxy columns in our features
relationship_cols = [c for c in features_test.columns if 'relationship' in c.lower()]
print(f"Relationship proxy columns found: {relationship_cols}")

# Create counterfactual: flip relationship proxies
# For each person, if relationship_Husband=1, set to 0 (and vice versa for Wife)
counterfactual_test = features_test.copy()

# Find the Wife and Husband columns (if they exist after one-hot encoding)
wife_col = [c for c in relationship_cols if 'wife' in c.lower()]
husband_col = [c for c in relationship_cols if 'husband' in c.lower()]

if wife_col and husband_col:
    wife_col = wife_col[0]
    husband_col = husband_col[0]
    
    # Swap Wife <-> Husband values
    original_wife = counterfactual_test[wife_col].copy()
    counterfactual_test[wife_col] = counterfactual_test[husband_col]
    counterfactual_test[husband_col] = original_wife
    print(f"Swapped {wife_col} <-> {husband_col}")
else:
    print("Wife/Husband columns not found - using a simpler perturbation")
    # Negate all relationship columns as a simple perturbation
    for col in relationship_cols:
        counterfactual_test[col] = -counterfactual_test[col]

print(f"Counterfactual test set created: {counterfactual_test.shape}")

## Step 3 — Measure prediction flip rate

How many predictions change when we flip the gender proxy features?

In [ ]:
counterfactual_predictions = baseline_model.predict(counterfactual_test)

# Compare original vs counterfactual predictions
prediction_flipped = original_predictions != counterfactual_predictions
flip_count = prediction_flipped.sum()
flip_rate = flip_count / len(prediction_flipped)

print(f"Total test samples:      {len(prediction_flipped)}")
print(f"Predictions that flipped: {flip_count}")
print(f"Flip rate:               {flip_rate:.2%}")
print()

if flip_rate > 0.10:
    print("[WARNING] HIGH sensitivity to gender proxy features!")
elif flip_rate > 0.05:
    print("[NOTICE] MODERATE sensitivity to gender proxy features.")
else:
    print("[OK] LOW sensitivity to gender proxy features.")

## Step 4 — Breakdown by original gender group

In [ ]:
flip_by_group = pd.DataFrame({
    "gender": gender_test.values,
    "flipped": prediction_flipped,
})

group_flip_rates = flip_by_group.groupby("gender")["flipped"].mean()
print("Flip rate by gender group:")
print(group_flip_rates)
print()

fig, ax = plt.subplots(figsize=(6, 4))
group_flip_rates.plot(kind="bar", ax=ax, color=["#FF9800", "#2196F3"], alpha=0.8)
ax.set_title("Counterfactual Flip Rate by Gender")
ax.set_ylabel("Flip Rate")
ax.set_xlabel("Original Gender")
ax.axhline(y=0.10, color='red', linestyle='--', alpha=0.5, label='10% threshold')
ax.legend()
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## Summary

- The counterfactual flip rate tells us how much the model relies on
  gender-proxy features (like relationship status).
- A high flip rate means the model's decisions are sensitive to gender,
  even though we removed `sex` from the features.
- This is complementary to the MetricFrame analysis — it tests a
  different dimension of fairness (individual vs. group).

**Next:** Notebook 06 — Benchmark Summary